# Bay of Bengal Sound-Speed Profile Reconstruction — Core Pipeline

This notebook reproduces the physics-constrained XGBoost model, the Mackenzie (1981) / TEOS-10 / EOF
baselines, and the LLM tactical-interpretation layer reported in the manuscript.

**Companion notebook:** `figures_and_extended_analysis.ipynb` contains the risk-map, seasonal, and
ray-tracing figures generated from this pipeline's outputs.

**Data:** This notebook expects `Bay_of_Bengal_Reconstructed_Full.csv` (compiled from Argo + World
Ocean Database + satellite products; see manuscript Section 2.1) in the working directory. Update
`CSV_PATH` below if your data lives elsewhere. Raw source data are not redistributed here — see the
Data Availability Statement in the manuscript for the original repositories.

**Environment:** see `requirements.txt` in the repository root.

**API key required:** the LLM interpretation layer (Section 3) calls the Groq API. Set the
`GROQ_API_KEY` environment variable before running that section — do not hardcode a key in this file.


## 0. Setup

In [ ]:
!pip install gsw groq xgboost shap scikit-learn pandas numpy matplotlib seaborn --quiet

import os
CSV_PATH = os.environ.get("SSP_DATA_PATH", "Bay_of_Bengal_Reconstructed_Full.csv")


## 1. Feature correlation check (CO2 as a secondary proxy)

Confirms that CO2 anomaly has weak direct correlation with `Actual_SSP`, consistent with its low
(0.4%) SHAP contribution reported in Section 4.4 — supporting evidence rather than a mechanistic
driver claim.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(CSV_PATH)

cols_of_interest = [
    "depth",
    "Salinity_Phys",
    "SST",
    "SSH",
    "CO2",
    "Actual_SSP",  # target
    # Pressure_db intentionally excluded: it is derived directly from depth
    # (r = 1.00, perfect collinearity) and is not an actual model input feature.
]

cols_present = [c for c in cols_of_interest if c in df.columns]
missing = [c for c in cols_of_interest if c not in df.columns]
if missing:
    print(f"Columns not found in CSV, skipped: {missing}")

corr_df = df[cols_present].corr(method="pearson")

print("=" * 60)
print("Correlation of CO2 with other variables:")
print("=" * 60)
co2_corr = corr_df["CO2"].sort_values(key=abs, ascending=False)
print(co2_corr)
print("\nCO2 vs Actual_SSP correlation:", corr_df.loc["CO2", "Actual_SSP"])

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_df, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0,
    square=True, linewidths=0.5, cbar_kws={"label": "Pearson Correlation Coefficient"},
)
plt.title("Feature Correlation Matrix — Bay of Bengal SSP Dataset", fontsize=11, pad=15)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("correlation_heatmap_CO2.png", dpi=300, bbox_inches="tight")
plt.show()


## 2. Main model + baselines (Table 2)

Trains XGBoost on the official 80/20 depth-stratified split (`random_state=42`) and evaluates it
against the Mackenzie (1981) empirical formula, TEOS-10, a climatological baseline, and a
profile-level EOF + ridge-regression baseline on the *same* held-out split — no data leakage between
baselines and the proposed model.

Note the split is stratified by depth quintile, not grouped by horizontal location; see the
Limitations discussion in Section 2.4 of the manuscript regarding partial train/test dependence
across depths at the same location.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

# ── 1. Load ──────────────────────────────────────────────────────────
print("Loading dataset...")
df = pd.read_csv(CSV_PATH)
print(f"Total samples: {len(df):,}")

# ── 2. Salinity -> Mackenzie term ───────────────────────────────────
S_raw = df["Salinity_Phys"].copy()  # raw PSU
df["Salinity_Phys_transformed"] = 1.34 * (S_raw - 35)

# ── 3. Mackenzie (1981) baseline ────────────────────────────────────
def mackenzie_ssp(T, S_raw_, z):
    return (1449.2 + 4.6*T - 0.055*T**2 + 0.00029*T**3 + 1.34*(S_raw_ - 35) + 0.016*z)

df["SSP_Mackenzie"] = mackenzie_ssp(df["Temp_Phys"], S_raw, df["depth"])

# ── 4. TEOS-10 baseline ──────────────────────────────────────────────
try:
    import gsw
    p_ = df["Pressure_db"].values
    sa_ = gsw.SA_from_SP(S_raw.values, p_, df["lon"].values, df["lat"].values)
    ct_ = gsw.CT_from_t(sa_, df["Temp_Phys"].values, p_)
    df["SSP_TEOS10"] = gsw.sound_speed(sa_, ct_, p_)
    teos10_ready = True
except ImportError:
    print("gsw not installed; skipping TEOS-10. Run: pip install gsw")
    teos10_ready = False

# ── 5. Cyclic day-of-year encoding ──────────────────────────────────
df["day_sin"] = np.sin(2*np.pi*df["day_of_year"]/365.0)
df["day_cos"] = np.cos(2*np.pi*df["day_of_year"]/365.0)

# ── 6. Features & target ────────────────────────────────────────────
df["Salinity_Phys"] = df["Salinity_Phys_transformed"]
features = ["depth", "Salinity_Phys", "SST", "SSH", "CO2", "day_sin", "day_cos"]
target = "Actual_SSP"

valid = df[features].notna().all(axis=1) & df[target].notna()
X = df[features][valid].reset_index(drop=True)
y = df[target][valid].reset_index(drop=True)
df_v = df[valid].reset_index(drop=True)
print(f"Usable samples: {len(X):,}")

# ── 7. Official train/test split ────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=pd.qcut(df_v["depth"], q=5, labels=False),
)
test_idx = X_test.index
train_idx = X_train.index
print(f"Test set size: {len(test_idx):,} (held-out test set)")

# ── 8. Train XGBoost ─────────────────────────────────────────────────
print("Training XGBoost...")
model = xgb.XGBRegressor(
    n_estimators=1500, max_depth=8, learning_rate=0.04,
    subsample=0.85, colsample_bytree=0.8, min_child_weight=5,
    random_state=42, tree_method="hist", verbosity=0,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
y_pred_xgb = model.predict(X_test)

# ── 9. Table 2 — all point-wise models on the same test_idx ────────
def metrics(y_true, y_pred, name):
    return {
        "Model": name,
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
    }

rows = [metrics(y_test, y_pred_xgb, "XGBoost (Proposed)")]
rows.append(metrics(y_test, df_v.loc[test_idx, "SSP_Mackenzie"], "Mackenzie (1981)"))
if teos10_ready:
    rows.append(metrics(y_test, df_v.loc[test_idx, "SSP_TEOS10"], "TEOS-10"))
rows.append(metrics(y_test, np.full(len(y_test), y_train.mean()), "Climatological Baseline"))

print("\n" + "="*65)
print(f"Table 2 — point-wise models, n={len(test_idx):,}")
print("="*65)
print(f"{'Model':<25}{'R2':>8}{'RMSE (m/s)':>13}{'MAE (m/s)':>12}")
print("-"*65)
for r in rows:
    print(f"{r['Model']:<25}{r['R2']:>8.4f}{r['RMSE']:>13.4f}{r['MAE']:>12.4f}")
print("="*65)

# ── 10. EOF baseline — leak-free, official split, profile-level ────
df_v["profile_id"] = (
    df_v["lat"].astype(str) + "_" + df_v["lon"].astype(str) + "_" + df_v["day_of_year"].astype(str)
)
df_v["split"] = "train"
df_v.loc[test_idx, "split"] = "test"

STANDARD_DEPTHS = np.arange(0, 301, 10)

def build_profile_matrix(sub_df, value_col="Actual_SSP"):
    rows_, ids_, preds_ = [], [], []
    for pid, g in sub_df.groupby("profile_id"):
        g = g.sort_values("depth")
        if g["depth"].nunique() < 5:
            continue
        vals = np.interp(STANDARD_DEPTHS, g["depth"].values, g[value_col].values)
        rows_.append(vals)
        ids_.append(pid)
        preds_.append(g[["SST", "SSH", "CO2"]].mean().values)
    matrix = pd.DataFrame(rows_, index=ids_, columns=STANDARD_DEPTHS)
    preds = pd.DataFrame(preds_, index=ids_, columns=["SST", "SSH", "CO2"])
    return matrix, preds

train_matrix, train_preds = build_profile_matrix(df_v[df_v["split"] == "train"])
test_matrix, test_preds = build_profile_matrix(df_v[df_v["split"] == "test"])

print(f"\nEOF training profiles (train split only): {len(train_matrix)}")
print(f"EOF evaluation profiles (official test split only): {len(test_matrix)}")

mean_profile = train_matrix.mean(axis=0).values
anomaly = train_matrix.values - mean_profile
n_modes = min(5, anomaly.shape[0]-1, anomaly.shape[1]-1)
U, Sv, Vt = np.linalg.svd(anomaly, full_matrices=False)
eofs = Vt[:n_modes]
pc_train = U[:, :n_modes] * Sv[:n_modes]

regs = []
for m in range(n_modes):
    r = Ridge(alpha=1.0)
    r.fit(train_preds.values, pc_train[:, m])
    regs.append(r)

pc_pred = np.column_stack([r.predict(test_preds.values) for r in regs])
recon = mean_profile + pc_pred @ eofs

y_true_eof = test_matrix.values.flatten()
y_pred_eof = recon.flatten()
eof_rmse = np.sqrt(mean_squared_error(y_true_eof, y_pred_eof))
eof_mae = mean_absolute_error(y_true_eof, y_pred_eof)
eof_r2 = r2_score(y_true_eof, y_pred_eof)

print("\n" + "="*65)
print(f"EOF regression baseline — official test-split profiles (n={len(test_matrix)})")
print("="*65)
print(f"RMSE: {eof_rmse:.4f}   MAE: {eof_mae:.4f}   R2: {eof_r2:.4f}")
print("="*65)
print("\nNote: the EOF profile count is lower than the row count (n=44,556)")
print("because EOF is evaluated profile-wise, not row-wise. This is by design.")


## 3. LLM tactical-interpretation layer (Section 3.5)

Generates a tactical summary for 1,000 stratified test-set samples (294 High Risk / 706 Safe Zone,
matching the manuscript's reported split) using Llama-3.1-8B-Instruct via the Groq API, given only
the top-3 SHAP feature attributions — the prompt does **not** tell the model which feature is
dominant; that is left for the model to identify.

**Requires `GROQ_API_KEY` to be set as an environment variable.** Get a key at
console.groq.com/keys. Never commit a key to this file.

In [ ]:
import os
import pandas as pd
import numpy as np
from groq import Groq
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
import warnings
import time
warnings.filterwarnings("ignore")

# ── 0. Groq client — reads key from environment, never hardcode it ──
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise RuntimeError(
        "Set the GROQ_API_KEY environment variable before running this cell, "
        "e.g. `export GROQ_API_KEY=...` or use a .env file with python-dotenv."
    )
client = Groq(api_key=GROQ_API_KEY)
MODEL_NAME = "llama-3.1-8b-instant"

# ── 1. Rebuild dataset + XGBoost + SHAP (same pipeline as Section 2) ─
df_csv = pd.read_csv(CSV_PATH)
S = df_csv["Salinity_Phys"].copy()
df_csv["Salinity_Phys"] = 1.34 * (S - 35)
df_csv["day_sin"] = np.sin(2 * np.pi * df_csv["day_of_year"] / 365.0)
df_csv["day_cos"] = np.cos(2 * np.pi * df_csv["day_of_year"] / 365.0)

features = ["depth", "Salinity_Phys", "SST", "SSH", "CO2", "day_sin", "day_cos"]
y = df_csv["Actual_SSP"]
valid = df_csv[features].notna().all(axis=1) & y.notna()
X = df_csv[features][valid].reset_index(drop=True)
y = y[valid].reset_index(drop=True)
df_v = df_csv[valid].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=pd.qcut(df_v["depth"], q=5, labels=False),
)

print("Training XGBoost...")
model_xgb = xgb.XGBRegressor(
    n_estimators=1500, max_depth=8, learning_rate=0.04,
    subsample=0.85, colsample_bytree=0.8, min_child_weight=5,
    random_state=42, tree_method="hist", verbosity=0,
)
model_xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred = model_xgb.predict(X_test)
errors = np.abs(y_test.values - y_pred)

THRESHOLD = 0.568
risk_class = np.where(errors >= THRESHOLD, "High Risk", "Safe Zone")

# ── 2. SHAP values (per-row top-3 features) ──────────────────────────
print("Computing SHAP values for test set...")
explainer = shap.TreeExplainer(model_xgb)
shap_values_test = explainer.shap_values(X_test)

def top3_shap_features(row_idx):
    vals = shap_values_test[row_idx]
    idx_sorted = np.argsort(-np.abs(vals))[:3]
    return [(features[i], vals[i]) for i in idx_sorted]

# ── 3. Sample 1000 profiles matching the reported 29.4% / 70.6% split ─
np.random.seed(42)
idx_high = np.where(risk_class == "High Risk")[0]
idx_safe = np.where(risk_class == "Safe Zone")[0]
n_high = min(294, len(idx_high))
n_safe = min(706, len(idx_safe))
sample_idx = np.concatenate([
    np.random.choice(idx_high, n_high, replace=False),
    np.random.choice(idx_safe, n_safe, replace=False),
])
np.random.shuffle(sample_idx)
print(f"Sampled {len(sample_idx)} profiles: {n_high} High Risk, {n_safe} Safe Zone")

# ── 4. Prompt construction — model is given SHAP values only, ────────
#      NOT told which feature is dominant; it must identify that itself.
SYSTEM_PROMPT = (
    "You are a naval oceanography analyst. You will be given the magnitude "
    "of a sound-speed-profile reconstruction error, the top-3 SHAP feature "
    "attributions driving that error, and a risk classification. Respond "
    "with EXACTLY two sentences: the first describing the dominant physical "
    "driver, the second stating the likely implication for underwater "
    "acoustic communication or submarine navigation. Be concise and technical."
)

def build_user_prompt(depth, error, risk, top3):
    shap_str = "; ".join([f"{name} (SHAP={val:+.3f})" for name, val in top3])
    return (
        f"Depth: {depth:.0f} m\n"
        f"Reconstruction error: {error:.3f} m/s\n"
        f"Risk classification: {risk}\n"
        f"Top-3 SHAP feature attributions: {shap_str}\n\n"
        f"Provide the two-sentence tactical summary."
    )

def generate_summary(depth, error, risk, top3, retries=3):
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": build_user_prompt(depth, error, risk, top3)},
                ],
                temperature=0.3,
                max_tokens=150,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}. Retrying...")
            time.sleep(2)
    return "[GENERATION FAILED after retries]"

# ── 5. Generate summaries for the 1000 sampled profiles ──────────────
results = []
print("\nGenerating tactical summaries via Groq (Llama-3.1-8B-Instruct)...")
for count, idx in enumerate(sample_idx):
    depth = X_test.iloc[idx]["depth"]
    error = errors[idx]
    risk = risk_class[idx]
    top3 = top3_shap_features(idx)

    summary = generate_summary(depth, error, risk, top3)

    results.append({
        "depth": depth,
        "error": error,
        "risk_class": risk,
        "top3_shap": top3,
        "llm_summary": summary,
    })

    if (count + 1) % 50 == 0:
        print(f"  {count+1}/{len(sample_idx)} summaries generated...")

    time.sleep(0.3)  # Groq free-tier rate limit

results_df = pd.DataFrame(results)
results_df.to_csv("llm_tactical_summaries.csv", index=False)
print(f"\nSaved {len(results_df)} LLM-generated summaries to llm_tactical_summaries.csv")


## 4. Automated SHAP-consistency check

Objective, reproducible check of whether each summary's first sentence names the feature that
actually has the largest-magnitude SHAP value (top-1) for that row. This is an **automated,
keyword-based check**, not a human evaluation — it verifies internal consistency between the LLM's
stated reasoning and the SHAP attribution, not the factual/oceanographic correctness of the
explanation itself.

In [ ]:
import pandas as pd
import numpy as np
import ast
import re

df = pd.read_csv("llm_tactical_summaries.csv")
print(f"Total summaries: {len(df)}")

FEATURE_KEYWORDS = {
    "depth": ["depth"],
    "Salinity_Phys": ["salinity"],
    "SST": ["temperature", "sst", "thermal"],
    "SSH": ["ssh", "sea surface height", "eddy"],
    "CO2": ["co2", "carbon dioxide"],
}

def get_top1_feature(top3_str):
    """Parses the top3_shap column and returns the largest-|value| feature."""
    try:
        cleaned = re.sub(r"np\.float32\(([^)]+)\)", r"\1", top3_str)
        top3 = ast.literal_eval(cleaned)
    except Exception:
        return None
    top3_sorted = sorted(top3, key=lambda x: -abs(x[1]))
    return top3_sorted[0][0]

def extract_claimed_driver(summary_text):
    """Returns which feature keyword(s) appear in the summary's first sentence."""
    first_sentence = summary_text.split(".")[0].lower()
    found = []
    for feature, keywords in FEATURE_KEYWORDS.items():
        for kw in keywords:
            if kw in first_sentence:
                found.append(feature)
                break
    return found

results = []
for idx, row in df.iterrows():
    top1 = get_top1_feature(row["top3_shap"])
    claimed = extract_claimed_driver(row["llm_summary"])

    if top1 is None:
        status = "PARSE_ERROR"
    elif len(claimed) == 0:
        status = "NO_FEATURE_MENTIONED"
    elif top1 in claimed and len(claimed) == 1:
        status = "MATCH"
    elif top1 in claimed and len(claimed) > 1:
        status = "MATCH_AMBIGUOUS"
    else:
        status = "MISMATCH"

    results.append({
        "depth": row["depth"],
        "risk_class": row["risk_class"],
        "top1_actual": top1,
        "claimed_features": claimed,
        "status": status,
    })

results_df = pd.DataFrame(results)
results_df.to_csv("automated_shap_consistency_check.csv", index=False)

print("\n" + "="*60)
print("AUTOMATED SHAP-CONSISTENCY CHECK RESULTS")
print("="*60)
status_counts = results_df["status"].value_counts()
print(status_counts)

n_match = status_counts.get("MATCH", 0)
n_match_ambig = status_counts.get("MATCH_AMBIGUOUS", 0)
n_mismatch = status_counts.get("MISMATCH", 0)
n_no_feature = status_counts.get("NO_FEATURE_MENTIONED", 0)
n_parse_err = status_counts.get("PARSE_ERROR", 0)

total_checked = len(results_df) - n_parse_err
strict_accuracy = 100 * n_match / total_checked
lenient_accuracy = 100 * (n_match + n_match_ambig) / total_checked

print(f"\nStrict accuracy (exact top-1 match only)             : {strict_accuracy:.1f}%")
print(f"Lenient accuracy (top-1 mentioned, even if ambiguous) : {lenient_accuracy:.1f}%")
print(f"Mismatch rate                                          : {100*n_mismatch/total_checked:.1f}%")
print(f"No clear feature mentioned                             : {n_no_feature}")


## 5. Heuristic content-validity screen on a 100-sample subset

**Important — please verify against the manuscript before release.** The scoring function below
(`auto_judge`) is a **rule-based keyword heuristic** (checks for words like "significant",
"catastrophic", "minimal", "safe", etc.), not a human domain-expert assessment. If the manuscript
describes this step as an "expert review" or implies manual/human evaluation, that wording should
be corrected to describe it accurately (e.g. "automated heuristic content screen"), or a genuine
human expert review should be conducted and reported instead, before this code and the manuscript
are made public together. The column name `Expert_Score` below has been kept only because it is
referenced by that name elsewhere; consider renaming it if you revise this script.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("llm_tactical_summaries.csv")
sample = df.sample(100, random_state=42).reset_index(drop=True)

def auto_judge(row):
    """Rule-based heuristic screen — NOT a human expert judgment."""
    summary = row["llm_summary"].lower()
    risk = row["risk_class"]

    if "depth" not in summary[:150]:
        return "n"
    if "bathymetry" in summary:
        return "n"
    if risk == "Safe Zone" and any(w in summary for w in
       ["significant impact", "catastrophic", "compromised", "malfunctions"]):
        return "p"
    if risk == "High Risk" and any(w in summary for w in
       ["minimal impact", "no impact", "negligible"]):
        return "n"
    if risk == "High Risk" and any(w in summary for w in
       ["compromised", "risk", "inaccurate", "error", "disruption"]):
        return "y"
    if risk == "Safe Zone" and any(w in summary for w in
       ["minimal", "low", "safe", "acceptable", "unaffected"]):
        return "y"
    return "p"

sample["Heuristic_Score"] = sample.apply(auto_judge, axis=1)

y = (sample["Heuristic_Score"] == "y").sum()
p = (sample["Heuristic_Score"] == "p").sum()
n = (sample["Heuristic_Score"] == "n").sum()

print(f"y: {y}/100 = {y}%")
print(f"p: {p}/100 = {p}%")
print(f"n: {n}/100 = {n}%")
print(f"Pass rate (y+p): {y+p}%")

sample.to_csv("heuristic_content_screen.csv", index=False)
